In [1]:
from pathlib import Path
import sys

import pandas as pd

analysis_dir = Path.cwd() if (Path.cwd() / "apple_health_xml.py").exists() else Path.cwd() / "src" / "analysis"
if str(analysis_dir) not in sys.path:
    sys.path.append(str(analysis_dir))

from apple_health_xml import apple_health_to_df, apple_health_type_counts

In [2]:
EXPORT_XML = next(path for path in [Path("health_data/export.xml"), Path("../../health_data/export.xml")] if path.exists())
EXPORT_XML.exists()

True

## Inspect available Apple Health record types

In [3]:
record_type_counts = apple_health_type_counts(EXPORT_XML, tag="Record")
record_type_counts.head(20)

,type,count,type_short
0,HKQuantityTypeIdentifierHeartRate,611661,HeartRate
1,HKQuantityTypeIdentifierActiveEnergyBurned,607203,ActiveEnergyBurned
2,HKQuantityTypeIdentifierBasalEnergyBurned,390027,BasalEnergyBurned
3,HKQuantityTypeIdentifierPhysicalEffort,349972,PhysicalEffort
4,HKQuantityTypeIdentifierStepCount,211026,StepCount
5,HKQuantityTypeIdentifierDistanceWalkingRunning,194891,DistanceWalkingRunning
6,HKQuantityTypeIdentifierAppleExerciseTime,76509,AppleExerciseTime
7,HKQuantityTypeIdentifierAppleStandTime,58898,AppleStandTime
8,HKQuantityTypeIdentifierWalkingSpeed,57220,WalkingSpeed
9,HKQuantityTypeIdentifierWalkingStepLength,57217,WalkingStepLength


## Convert records to DataFrames

In [4]:
# Load one metric while exploring. Remove type_filter to load all ~millions of records.
steps_df = apple_health_to_df(
    EXPORT_XML,
    tag="Record",
    type_filter="HKQuantityTypeIdentifierStepCount",
)
steps_df.head()

,type,sourceName,sourceVersion,device,unit,creationDate,startDate,endDate,value,value_numeric,type_short
0,HKQuantityTypeIdentifierStepCount,Claudio’s Big Brick,15.0.1,"<<HKDevice: 0xc1d5914a0>, name:iPhone, manufac...",count,2021-11-02 05:49:39+10:00,2021-11-02 05:38:31+10:00,2021-11-02 05:48:23+10:00,20,20,StepCount
1,HKQuantityTypeIdentifierStepCount,Claudio’s Big Brick,15.0.1,"<<HKDevice: 0xc1d5914a0>, name:iPhone, manufac...",count,2021-11-02 06:11:41+10:00,2021-11-02 06:00:38+10:00,2021-11-02 06:00:48+10:00,17,17,StepCount
2,HKQuantityTypeIdentifierStepCount,Claudio’s Big Brick,15.0.1,"<<HKDevice: 0xc1d5914a0>, name:iPhone, manufac...",count,2021-11-02 06:23:17+10:00,2021-11-02 06:12:14+10:00,2021-11-02 06:14:32+10:00,17,17,StepCount
3,HKQuantityTypeIdentifierStepCount,Claudio’s Big Brick,15.0.1,"<<HKDevice: 0xc1d5914a0>, name:iPhone, manufac...",count,2021-11-02 08:12:22+10:00,2021-11-02 08:09:07+10:00,2021-11-02 08:09:27+10:00,22,22,StepCount
4,HKQuantityTypeIdentifierStepCount,Claudio’s Big Brick,15.0.1,"<<HKDevice: 0xc1d5914a0>, name:iPhone, manufac...",count,2021-11-02 21:23:15+10:00,2021-11-02 21:17:39+10:00,2021-11-02 21:17:44+10:00,9,9,StepCount


In [5]:
# Example for multiple metric types.
vitals_df = apple_health_to_df(
    EXPORT_XML,
    tag="Record",
    type_filter={
        "HKQuantityTypeIdentifierHeartRate",
        "HKQuantityTypeIdentifierRestingHeartRate",
        "HKQuantityTypeIdentifierWalkingHeartRateAverage",
    },
)
vitals_df.head()

,type,sourceName,sourceVersion,device,unit,creationDate,startDate,endDate,value,value_numeric,type_short
0,HKQuantityTypeIdentifierHeartRate,王阳明’s Apple Watch,8.1,"<<HKDevice: 0xc1d593de0>, name:Apple Watch, ma...",count/min,2022-01-09 09:44:07+10:00,2022-01-09 09:36:29+10:00,2022-01-09 09:36:29+10:00,76,76.0,HeartRate
1,HKQuantityTypeIdentifierHeartRate,王阳明’s Apple Watch,8.1,"<<HKDevice: 0xc1d593de0>, name:Apple Watch, ma...",count/min,2022-01-09 09:55:36+10:00,2022-01-09 09:53:25+10:00,2022-01-09 09:53:25+10:00,68,68.0,HeartRate
2,HKQuantityTypeIdentifierHeartRate,王阳明’s Apple Watch,8.1,"<<HKDevice: 0xc1d593de0>, name:Apple Watch, ma...",count/min,2022-01-09 10:00:16+10:00,2022-01-09 09:58:54+10:00,2022-01-09 09:58:54+10:00,62,62.0,HeartRate
3,HKQuantityTypeIdentifierHeartRate,王阳明’s Apple Watch,8.1,"<<HKDevice: 0xc1d593de0>, name:Apple Watch, ma...",count/min,2022-01-09 10:05:21+10:00,2022-01-09 10:02:10+10:00,2022-01-09 10:02:10+10:00,64,64.0,HeartRate
4,HKQuantityTypeIdentifierHeartRate,王阳明’s Apple Watch,8.1,"<<HKDevice: 0xc1d593de0>, name:Apple Watch, ma...",count/min,2022-01-09 10:12:31+10:00,2022-01-09 10:05:41+10:00,2022-01-09 10:05:41+10:00,65,65.0,HeartRate


## Convert workouts and daily activity summaries

In [6]:
workouts_df = apple_health_to_df(EXPORT_XML, tag="Workout", include_metadata=True)
workouts_df.head()

,workoutActivityType,duration,durationUnit,sourceName,sourceVersion,device,creationDate,startDate,endDate,metadata_HKIndoorWorkout,...,stat_RunningStrideLength_average_numeric,stat_BasalEnergyBurned_sum_numeric,stat_HeartRate_maximum_numeric,stat_DistanceCycling_sum_numeric,stat_RunningSpeed_minimum_numeric,stat_HeartRate_average_numeric,stat_RunningGroundContactTime_maximum_numeric,duration_numeric,stat_HeartRate_minimum_numeric,workoutActivityType_short
0,HKWorkoutActivityTypeCycling,10.23855701287587,min,王阳明’s Apple Watch,8.3,"<<HKDevice: 0xc1e2cb300>, name:Apple Watch, ma...",2022-01-14 23:08:20+10:00,2022-01-14 22:58:05+10:00,2022-01-14 23:08:19+10:00,0,...,NaN,13.17350,NaN,1.51658,NaN,NaN,NaN,10.238557,NaN,Cycling
1,HKWorkoutActivityTypeRunning,2.313007865349452,min,王阳明’s Apple Watch,8.3,"<<HKDevice: 0xc1e2cb300>, name:Apple Watch, ma...",2022-01-23 05:39:02+10:00,2022-01-23 05:36:42+10:00,2022-01-23 05:39:01+10:00,0,...,NaN,2.93250,NaN,NaN,NaN,NaN,NaN,2.313008,NaN,Running
2,HKWorkoutActivityTypeFunctionalStrengthTraining,9.418020983537039,min,王阳明’s Apple Watch,8.4,"<<HKDevice: 0xc1e2cbcc0>, name:Apple Watch, ma...",2022-01-30 07:11:48+10:00,2022-01-30 07:02:22+10:00,2022-01-30 07:11:47+10:00,0,...,NaN,12.96360,NaN,NaN,NaN,NaN,NaN,9.418021,NaN,FunctionalStrengthTraining
3,HKWorkoutActivityTypeFunctionalStrengthTraining,61.05593038400014,min,王阳明’s Apple Watch,8.4,"<<HKDevice: 0xc1e2cbcc0>, name:Apple Watch, ma...",2022-01-30 08:26:52+10:00,2022-01-30 07:25:48+10:00,2022-01-30 08:26:52+10:00,0,...,NaN,84.82620,NaN,NaN,NaN,NaN,NaN,61.055930,NaN,FunctionalStrengthTraining
4,HKWorkoutActivityTypeRunning,3.50838913321495,min,王阳明’s Apple Watch,8.4,"<<HKDevice: 0xc1e2cbcc0>, name:Apple Watch, ma...",2022-01-31 05:02:31+10:00,2022-01-31 04:59:00+10:00,2022-01-31 05:02:31+10:00,0,...,NaN,4.72552,NaN,NaN,NaN,NaN,NaN,3.508389,NaN,Running


In [7]:
activity_summary_df = apple_health_to_df(EXPORT_XML, tag="ActivitySummary")
activity_summary_df.head()

,dateComponents,activeEnergyBurned,activeEnergyBurnedGoal,activeEnergyBurnedUnit,appleMoveTime,appleMoveTimeGoal,appleExerciseTime,appleExerciseTimeGoal,appleStandHours,appleStandHoursGoal,activeEnergyBurned_numeric,appleExerciseTime_numeric,appleMoveTimeGoal_numeric,appleStandHours_numeric,appleStandHoursGoal_numeric,appleMoveTime_numeric,appleExerciseTimeGoal_numeric,activeEnergyBurnedGoal_numeric
0,2022-01-07,0,0,kcal,0,0,0,30,0,12,0.000,0,0,0,12,0,30,0
1,2022-01-08,0,0,kcal,0,0,0,30,0,12,0.000,0,0,0,12,0,30,0
2,2022-01-09,222.344,330,kcal,0,0,17,30,5,6,222.344,17,0,5,6,0,30,330
3,2022-01-10,293.091,330,kcal,0,0,23,30,7,6,293.091,23,0,7,6,0,30,330
4,2022-01-11,93.821,330,kcal,0,0,8,30,3,6,93.821,8,0,3,6,0,30,330
